# ESCUELA POLITECNICA NACIONAL

**INTEGRANTES:**
* Goyes
* Jiménez Freddy
* Ochoa Aubertin
* Romero Erick

**FECHA:** 23/06/2026


# PROYECTO: ETL - TRANSPORTE PÚBLICO QUITO (2024)
### EQUIPO #8:
### OBJETIVO: Extraer, transformar y cargar (ETL) los datos de validaciones al Data Warehouse.

In [5]:
# --- 1. INSTALAR Y IMPORTAR LIBRERÍAS ---
import pandas as pd
import re
import logging
from datetime import datetime, timedelta
import os

In [6]:
# --- 2. CONFIGURACIÓN DE LOGS (Control de errores) ---
logging.basicConfig(
    filename='etl_transporte_errores.log',
    level=logging.ERROR,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
print("Configuración de logs iniciada.")

Configuración de logs iniciada.


In [7]:
# --- 3. FUNCIONES DE AYUDA (PARSEO Y TRANSFORMACIÓN) ---

def extraer_codigo_y_nombre(texto):
    """
    Extrae el código entre paréntesis y el nombre descriptivo.
    Ej: "(10013) M84 - C84" -> ("10013", "M84 - C84")
    """
    if pd.isna(texto):
        return None, None
    texto = str(texto).strip()
    match = re.match(r'\(([^)]+)\)\s*(.*)', texto)
    if match:
        return match.group(1).strip(), match.group(2).strip()
    return None, texto

def parsear_intervalo(intervalo_str):
    """
    Convierte "6:15" en hora_inicio (TIME) y hora_fin (TIME + 15 min)
    """
    if pd.isna(intervalo_str):
        return None, None
    try:
        parts = str(intervalo_str).strip().split(':')
        hora = int(parts[0])
        minuto = int(parts[1])
        inicio = datetime.strptime(f"{hora:02d}:{minuto:02d}", "%H:%M").time()
        fin_dt = datetime.strptime(f"{hora:02d}:{minuto:02d}", "%H:%M") + timedelta(minutes=15)
        fin = fin_dt.time()
        return inicio, fin
    except Exception as e:
        logging.error(f"Error parseando intervalo: {intervalo_str} - {e}")
        return None, None

def asignar_franja(hora_inicio):
    """Asigna franja horaria según la hora de inicio."""
    if hora_inicio is None:
        return "Desconocido"
    h = hora_inicio.hour
    if 4 <= h < 6:
        return "Madrugada"
    elif 6 <= h < 12:
        return "Mañana"
    elif 12 <= h < 18:
        return "Tarde"
    elif 18 <= h < 24:
        return "Noche"
    else:
        return "Desconocido"

In [9]:
# ============================================================
# --- 4. EXTRACCIÓN (EXTRACT) - LEER ARCHIVO EXCEL LOCAL ---
# ============================================================
print("="*60)
print("PROCESO ETL - TRANSPORTE PÚBLICO QUITO")
print("="*60)

# NOMBRE DEL ARCHIVO (ajústalo si es necesario)
FILE_PATH = 'ValidacionesAbr2024.xlsx'   # Tu archivo Excel
SHEET_NAME = 'Validaciones Tullave'      # Nombre de la hoja principal

try:
    # Leer el archivo Excel
    df_staging = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME)
    print(f"\nEXTRACCIÓN EXITOSA: {df_staging.shape[0]} filas, {df_staging.shape[1]} columnas.")
except FileNotFoundError:
    print(f"\nERROR: No se encontró el archivo '{FILE_PATH}'.")
    exit()
except Exception as e:
    print(f"\nERROR AL LEER ARCHIVO: {e}")
    exit()

# Mostrar primeras filas para validar
print("\nVista previa de los datos extraídos:")
print(df_staging.head(3))
print(f"\nColumnas detectadas: {df_staging.columns.tolist()}")
print(f"Tipos de datos:\n{df_staging.dtypes}")

PROCESO ETL - TRANSPORTE PÚBLICO QUITO

EXTRACCIÓN EXITOSA: 78522 filas, 37 columnas.

Vista previa de los datos extraídos:
   Fase            Línea           Estación Acceso de Estación Intervalo  \
0  Dual  (10006) M84-C84  (10013) M84 - C84      (0) (Unknown)  06:15:00   
1  Dual  (10006) M84-C84  (10013) M84 - C84      (0) (Unknown)  07:00:00   
2  Dual  (10006) M84-C84  (10013) M84 - C84      (0) (Unknown)  07:15:00   

   2024-04-01 00:00:00  2024-04-02 00:00:00  2024-04-03 00:00:00  \
0                  0.0                 43.0                  0.0   
1                  0.0                  1.0                  0.0   
2                  0.0                  2.0                  0.0   

   2024-04-04 00:00:00  2024-04-05 00:00:00  ...  2024-04-23 00:00:00  \
0                  0.0                  0.0  ...                  0.0   
1                  0.0                  0.0  ...                  0.0   
2                  0.0                  0.0  ...                  0.0   

   20

In [10]:
# ============================================================
# --- 5. TRANSFORMACIÓN (TRANSFORM) ---
# ============================================================
print("\nIniciando fase de TRANSFORMACIÓN...")

# 5.1. Identificar columnas de fechas
atributos = ['Fase', 'Línea', 'Estación', 'Acceso de Estación', 'Intervalo']
fecha_columns = [col for col in df_staging.columns if col not in atributos and col != 'Total general']
print(f"Columnas de fecha detectadas: {len(fecha_columns)} días.")

# 5.2. UNPIVOT
df_melted = pd.melt(
    df_staging,
    id_vars=atributos,
    value_vars=fecha_columns,
    var_name='fecha_str',
    value_name='total_validaciones'
)
print(f"Registros después de UNPIVOT: {df_melted.shape[0]}")

# 5.3. Limpieza
df_melted = df_melted[df_melted['total_validaciones'] > 0]
df_melted = df_melted.dropna(subset=['total_validaciones'])
df_melted['total_validaciones'] = df_melted['total_validaciones'].astype(int)

# 5.4. Parsear Fecha
df_melted['fecha'] = pd.to_datetime(df_melted['fecha_str'], format='%d/%m/%Y', errors='coerce')
df_melted = df_melted.dropna(subset=['fecha'])

# 5.5. Crear dimensiones
df_melted['codigo_fase'] = df_melted['Fase'].str.strip().str.upper()
df_melted['nombre_fase'] = df_melted['Fase'].str.strip()

df_melted[['codigo_linea', 'nombre_linea']] = df_melted['Línea'].apply(
    lambda x: pd.Series(extraer_codigo_y_nombre(x))
)
df_melted[['codigo_estacion', 'nombre_estacion']] = df_melted['Estación'].apply(
    lambda x: pd.Series(extraer_codigo_y_nombre(x))
)
df_melted[['codigo_acceso', 'nombre_acceso']] = df_melted['Acceso de Estación'].apply(
    lambda x: pd.Series(extraer_codigo_y_nombre(x))
)

df_melted[['hora_inicio', 'hora_fin']] = df_melted['Intervalo'].apply(
    lambda x: pd.Series(parsear_intervalo(x))
)
df_melted['intervalo_texto'] = df_melted['Intervalo'].astype(str).str.strip()
df_melted['franja_horaria'] = df_melted['hora_inicio'].apply(asignar_franja)

# Dimensión Tiempo
df_melted['dia'] = df_melted['fecha'].dt.day
df_melted['mes'] = df_melted['fecha'].dt.month
month_map = {1:'Enero',2:'Febrero',3:'Marzo',4:'Abril',5:'Mayo',6:'Junio',
             7:'Julio',8:'Agosto',9:'Septiembre',10:'Octubre',11:'Noviembre',12:'Diciembre'}
df_melted['nombre_mes'] = df_melted['mes'].map(month_map)
df_melted['anio'] = df_melted['fecha'].dt.year
df_melted['dia_semana'] = df_melted['fecha'].dt.isocalendar().day
df_melted['nombre_dia'] = df_melted['fecha'].dt.day_name()
day_map = {'Monday':'Lunes','Tuesday':'Martes','Wednesday':'Miércoles',
           'Thursday':'Jueves','Friday':'Viernes','Saturday':'Sábado','Sunday':'Domingo'}
df_melted['nombre_dia'] = df_melted['nombre_dia'].map(day_map)
df_melted['fin_semana'] = df_melted['dia_semana'].apply(lambda x: 1 if x >= 5 else 0)

# 5.6. Control de errores
df_errores = df_melted[
    df_melted['codigo_linea'].isna() |
    df_melted['codigo_estacion'].isna() |
    df_melted['codigo_acceso'].isna() |
    df_melted['hora_inicio'].isna()
]
if not df_errores.empty:
    print(f"ATENCIÓN: {df_errores.shape[0]} registros con datos incompletos.")
    df_errores.to_csv('errores_etl.csv', index=False)
    logging.error(f"Se encontraron {df_errores.shape[0]} registros inválidos.")

df_clean = df_melted.dropna(subset=['codigo_linea', 'codigo_estacion', 'codigo_acceso', 'hora_inicio'])
print(f"Registros limpios listos para cargar: {df_clean.shape[0]}")


Iniciando fase de TRANSFORMACIÓN...
Columnas de fecha detectadas: 31 días.
Registros después de UNPIVOT: 2434182
ATENCIÓN: 30 registros con datos incompletos.
Registros limpios listos para cargar: 883958


In [11]:
# ============================================================
# --- 6. CARGA (LOAD) - GUARDAR EN CSVs ---
# ============================================================
print("\nIniciando fase de CARGA (guardando CSVs)...")

# Dimensión Tiempo
dim_tiempo = df_clean[['fecha', 'dia', 'mes', 'nombre_mes', 'anio', 'dia_semana', 'nombre_dia', 'fin_semana']].drop_duplicates('fecha').copy()
dim_tiempo['id_fecha'] = range(1, len(dim_tiempo) + 1)
dim_tiempo.to_csv('Dim_Tiempo.csv', index=False)
print(f"Dim_Tiempo: {len(dim_tiempo)} registros.")

# Dimensión Fase
dim_fase = df_clean[['codigo_fase', 'nombre_fase']].drop_duplicates('codigo_fase').copy()
dim_fase['id_fase'] = range(1, len(dim_fase) + 1)
dim_fase.to_csv('Dim_Fase.csv', index=False)
print(f"Dim_Fase: {len(dim_fase)} registros.")

# Dimensión Línea
dim_linea = df_clean[['codigo_linea', 'nombre_linea']].drop_duplicates('codigo_linea').copy()
dim_linea['id_linea'] = range(1, len(dim_linea) + 1)
dim_linea.to_csv('Dim_Linea.csv', index=False)
print(f"Dim_Linea: {len(dim_linea)} registros.")

# Dimensión Estación
dim_estacion = df_clean[['codigo_estacion', 'nombre_estacion']].drop_duplicates('codigo_estacion').copy()
dim_estacion['id_estacion'] = range(1, len(dim_estacion) + 1)
dim_estacion.to_csv('Dim_Estacion.csv', index=False)
print(f"Dim_Estacion: {len(dim_estacion)} registros.")

# Dimensión Acceso
dim_acceso = df_clean[['codigo_acceso', 'nombre_acceso']].drop_duplicates('codigo_acceso').copy()
dim_acceso['id_acceso'] = range(1, len(dim_acceso) + 1)
dim_acceso.to_csv('Dim_Acceso.csv', index=False)
print(f"Dim_Acceso: {len(dim_acceso)} registros.")

# Dimensión Intervalo
dim_intervalo = df_clean[['hora_inicio', 'hora_fin', 'intervalo_texto', 'franja_horaria']].drop_duplicates('intervalo_texto').copy()
dim_intervalo['id_intervalo'] = range(1, len(dim_intervalo) + 1)
dim_intervalo['hora_inicio'] = dim_intervalo['hora_inicio'].astype(str)
dim_intervalo['hora_fin'] = dim_intervalo['hora_fin'].astype(str)
dim_intervalo.to_csv('Dim_Intervalo_Horario.csv', index=False)
print(f"Dim_Intervalo_Horario: {len(dim_intervalo)} registros.")

# FACT_VALIDACIONES
fact = df_clean.merge(dim_tiempo[['fecha', 'id_fecha']], on='fecha', how='left')
fact = fact.merge(dim_fase[['codigo_fase', 'id_fase']], on='codigo_fase', how='left')
fact = fact.merge(dim_linea[['codigo_linea', 'id_linea']], on='codigo_linea', how='left')
fact = fact.merge(dim_estacion[['codigo_estacion', 'id_estacion']], on='codigo_estacion', how='left')
fact = fact.merge(dim_acceso[['codigo_acceso', 'id_acceso']], on='codigo_acceso', how='left')
fact = fact.merge(dim_intervalo[['intervalo_texto', 'id_intervalo']], on='intervalo_texto', how='left')

fact_final = fact[['id_fecha', 'id_fase', 'id_linea', 'id_estacion', 'id_acceso', 'id_intervalo', 'total_validaciones']].dropna()
fact_final['id_validacion'] = range(1, len(fact_final) + 1)
fact_final = fact_final[['id_validacion', 'id_fecha', 'id_fase', 'id_linea', 'id_estacion', 'id_acceso', 'id_intervalo', 'total_validaciones']]
fact_final.to_csv('Fact_Validaciones.csv', index=False)
print(f"Fact_Validaciones: {len(fact_final)} registros.")

print("\nPROCESO ETL COMPLETADO CON ÉXITO.")
print(f"Total de validaciones procesadas: {fact_final['total_validaciones'].sum():,}")

print("\nArchivos generados:")
print("   - Dim_Tiempo.csv")
print("   - Dim_Fase.csv")
print("   - Dim_Linea.csv")
print("   - Dim_Estacion.csv")
print("   - Dim_Acceso.csv")
print("   - Dim_Intervalo_Horario.csv")
print("   - Fact_Validaciones.csv")


Iniciando fase de CARGA (guardando CSVs)...
Dim_Tiempo: 30 registros.
Dim_Fase: 4 registros.
Dim_Linea: 21 registros.
Dim_Estacion: 176 registros.
Dim_Acceso: 201 registros.
Dim_Intervalo_Horario: 95 registros.
Fact_Validaciones: 883958 registros.

PROCESO ETL COMPLETADO CON ÉXITO.
Total de validaciones procesadas: 51,366,659

Archivos generados:
   - Dim_Tiempo.csv
   - Dim_Fase.csv
   - Dim_Linea.csv
   - Dim_Estacion.csv
   - Dim_Acceso.csv
   - Dim_Intervalo_Horario.csv
   - Fact_Validaciones.csv
